# Faster R-CNN Experiments: GOST Stamp Detection

**Цель:** Обучить Faster R-CNN для детекции штампов на строительных чертежах.

**Данные:** 500 synthetic (train) + 49 real (val)

**Метрики:** IoU, Precision, Recall, F1 на 49 реальных изображениях

**Подход:** Single training run, ResNet50 FPN backbone, 30 epochs, GPU T4 (Colab)


## 0. Colab Setup

⚠️ **Запустить только один раз!** Клонирует репозиторий (sparse checkout) и устанавливает зависимости.

In [ ]:
%%bash
cd /content
rm -rf aie-group-2-sapar
git init aie-group-2-sapar
cd aie-group-2-sapar
git sparse-checkout set project
git remote add origin https://github.com/Sapar-hub/aie-group-2-sapar.git
git pull origin main
cd project
pip install -q torch torchvision opencv-python-headless pyyaml

In [ ]:
%cd /content/aie-group-2-sapar/project

In [ ]:
!nvidia-smi

## 1. Generate Synthetic Data

⚠️ **Запустить только один раз!** Генерирует 500 синтетических изображений.

In [ ]:
!python scripts/generate_synthetic.py --output data/ --num-gost 250 --num-copy 250 --dpi 200

## 2. Imports & Data Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / "src"))

import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F
import numpy as np
import matplotlib.pyplot as plt
import cv2
from torch.utils.data import Dataset, DataLoader

from evaluation.metrics import DetectionResult, bbox_iou, yolo_to_pixel, compute_metrics, print_metrics
from data.loader import load_image_and_labels

DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)
(ARTIFACTS_DIR / "models").mkdir(exist_ok=True)
(ARTIFACTS_DIR / "metrics").mkdir(exist_ok=True)
(ARTIFACTS_DIR / "figures").mkdir(exist_ok=True)

IMAGE_TEST_DIR = DATA_DIR / "images" / "test"
LABEL_TEST_DIR = DATA_DIR / "labels" / "test"
IMAGE_TRAIN_DIR = DATA_DIR / "images" / "train"
LABEL_TRAIN_DIR = DATA_DIR / "labels" / "train"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Train images: {len(list(IMAGE_TRAIN_DIR.glob('*.png')))}")
print(f"Test images: {len(list(IMAGE_TEST_DIR.glob('*.png')))}")

## 3. Dataset & DataLoader

In [ ]:
MAX_SIZE = 800

def resize_with_boxes(image, boxes, max_size=MAX_SIZE):
    """Resize image so longest side <= max_size, adjust boxes accordingly."""
    h, w = image.shape[:2]
    scale = max_size / max(h, w)
    if scale >= 1.0:
        return image, boxes, 1.0
    new_w, new_h = int(w * scale), int(h * scale)
    image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    if len(boxes) > 0:
        boxes = boxes * scale
    return image, boxes, scale

class GOSTDataset(Dataset):
    def __init__(self, image_dir, label_dir, max_size=MAX_SIZE):
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir)
        self.max_size = max_size
        self.image_paths = sorted(list(self.image_dir.glob("*.png")) + list(self.image_dir.glob("*.jpg")))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        label_files = list(self.label_dir.glob(f"*-{img_path.stem}.txt"))
        if label_files:
            with open(label_files[0]) as f:
                labels = [list(map(float, line.strip().split())) for line in f if line.strip()]
            boxes = []
            for label in labels:
                _, cx, cy, bw, bh = label
                x1 = (cx - bw / 2) * w
                y1 = (cy - bh / 2) * h
                x2 = (cx + bw / 2) * w
                y2 = (cy + bh / 2) * h
                boxes.append([x1, y1, x2, y2])
            boxes = torch.tensor(boxes, dtype=torch.float32)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)

        img, boxes, _ = resize_with_boxes(img, boxes, self.max_size)
        img_tensor = F.to_tensor(img)

        labels_tensor = torch.ones(len(boxes), dtype=torch.int64) if len(boxes) > 0 else torch.zeros(0, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels_tensor}
        return img_tensor, target

train_dataset = GOSTDataset(IMAGE_TRAIN_DIR, LABEL_TRAIN_DIR)
test_dataset = GOSTDataset(IMAGE_TEST_DIR, LABEL_TEST_DIR)
print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")

## 4. Training

In [ ]:
import time
start = time.time()

NUM_CLASSES = 2
NUM_EPOCHS = 30
BATCH_SIZE = 8
LR = 0.001

model = fasterrcnn_resnet50_fpn(weights=None)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, NUM_CLASSES)
model.to(DEVICE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, collate_fn=lambda x: tuple(zip(*x)), num_workers=0, pin_memory=True)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=LR, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

loss_history = []
for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0
    for images, targets in train_loader:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        epoch_loss += losses.item()
    lr_scheduler.step()
    loss_history.append(epoch_loss / len(train_loader))
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {loss_history[-1]:.4f}")

elapsed = time.time() - start
print(f"\nTraining time: {elapsed/60:.1f} minutes")

torch.save(model.state_dict(), str(ARTIFACTS_DIR / "models" / "rcnn_best.pth"))
print(f"Weights saved to {ARTIFACTS_DIR / 'models' / 'rcnn_best.pth'}")

## 5. Loss Plot

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, NUM_EPOCHS+1), loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Faster R-CNN Training Loss")
plt.grid(True, alpha=0.3)
plt.savefig(ARTIFACTS_DIR / "figures" / "rcnn_loss.png", dpi=150)
plt.show()

## 6. Evaluation on 49 Real Images

In [ ]:
model.eval()
results_list = []

all_images = sorted(IMAGE_TEST_DIR.glob("*.png")) + sorted(IMAGE_TEST_DIR.glob("*.jpg"))
for img_path in all_images:
    img, labels = load_image_and_labels(img_path, LABEL_TEST_DIR)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    gt_bbox = yolo_to_pixel(tuple(labels[0]), w, h) if len(labels) > 0 else None
    
    # Resize for inference (same as training)
    scale = MAX_SIZE / max(h, w)
    if scale < 1.0:
        new_w, new_h = int(w * scale), int(h * scale)
        img_rgb = cv2.resize(img_rgb, (new_w, new_h), interpolation=cv2.INTER_AREA)
        if gt_bbox:
            x, y, bw, bh = gt_bbox
            gt_bbox = (int(x * scale), int(y * scale), int(bw * scale), int(bh * scale))
    
    img_tensor = F.to_tensor(img_rgb).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        preds = model(img_tensor)[0]
    
    boxes = preds["boxes"].cpu().numpy()
    scores = preds["scores"].cpu().numpy()
    mask = scores >= 0.3
    
    if mask.any():
        best_idx = mask.argmax()
        x1, y1, x2, y2 = map(int, boxes[best_idx])
        pred_bbox = (x1, y1, x2 - x1, y2 - y1)
    else:
        pred_bbox = None
    
    iou = bbox_iou(pred_bbox, gt_bbox) if pred_bbox and gt_bbox else 0.0
    results_list.append(DetectionResult(
        image_name=img_path.name,
        gt_bbox=gt_bbox,
        pred_bbox=pred_bbox,
        iou=iou,
        found=pred_bbox is not None
    ))

metrics = compute_metrics(results_list, iou_threshold=0.5)
print_metrics(metrics, prefix="RCNN ")

## 7. Visualization

In [ ]:
sorted_results = sorted(results_list, key=lambda r: r.iou)
worst = sorted_results[0]
best = sorted_results[-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, result, title_prefix in zip(axes, [worst, best], ["Worst", "Best"]):
    img_path = [p for p in all_images if p.name == result.image_name][0]
    img, _ = load_image_and_labels(img_path, LABEL_TEST_DIR)
    vis = img.copy()
    if result.gt_bbox:
        x, y, bw, bh = result.gt_bbox
        cv2.rectangle(vis, (x, y), (x+bw, y+bh), (0, 255, 0), 3)
    if result.pred_bbox:
        x, y, bw, bh = result.pred_bbox
        cv2.rectangle(vis, (x, y), (x+bw, y+bh), (0, 0, 255), 2)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{title_prefix} IoU={result.iou:.3f}")
    ax.axis("off")

plt.suptitle("Green=GT, Red=Pred (Faster R-CNN)")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "figures" / "rcnn_best_worst.png", dpi=150)
plt.show()

## 8. Conclusions

In [ ]:
summary = {
    "model": "Faster R-CNN (ResNet50 FPN)",
    "data": "500 synthetic + 49 real (val)",
    "epochs": NUM_EPOCHS,
    "train_time_min": round(elapsed/60, 1),
    "iou_mean": round(metrics['iou_mean'], 3),
    "iou_std": round(metrics['iou_std'], 3),
    "precision": round(metrics['precision'], 3),
    "recall": round(metrics['recall'], 3),
    "f1": round(metrics['f1'], 3),
    "detection_rate": round(metrics['detection_rate'], 3),
}

import json
with open(ARTIFACTS_DIR / "metrics" / "rcnn_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Summary saved to artifacts/metrics/rcnn_results.json")
print(json.dumps(summary, indent=2))

## 9. Save Results to Git

⚠️ **Запустить после обучения!** Сохраняет метрики и графики в репозиторий.

In [ ]:
%%bash
cd /content/aie-group-2-sapar
git config user.email "183649607+Sapar-hub@users.noreply.github.com"
git config user.name "Saparmyrat"
git add project/artifacts/metrics/ project/artifacts/figures/
git commit -m "exp05: Faster R-CNN results and metrics"
git push origin main